In [5]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import ast

In [2]:
ftpred = pd.read_csv('private_data/ft_db_cbcts_dummy_vit-t_ftdet.csv')
ftpred_train = ftpred[ftpred['subsplit'] == 'train']
ftpred_val = ftpred[ftpred['subsplit'] == 'val']
ftpred_test = ftpred[ftpred['subsplit'] == 'test']

In [3]:
class FTPredDataset(Dataset):
    def __init__(self,
                 df: pd.DataFrame,
                 crop_or_pad_to: int = 30,
                 transforms = None):
        self.df = df
        self.transform = transforms
        self.groups = [g for _, g in self.df.groupby("id")]
        self.crop_or_pad_to = crop_or_pad_to

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        sample = self.groups[idx]

        if self.transform:
            sample = self.transform(sample)
        X = sample.drop(columns=["event", "event_at_time", "time", "start", "stop", "id", "split", "subsplit"]).values
        X = self.get_padded_features(X)
        Y = (sample['time'] - sample['stop']).values[-1] # get last
        D = sample['event_at_time'].values[-1] # get last
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(Y, dtype=torch.float32),
                torch.tensor(D, dtype=torch.float32))

    def get_padded_features(self, x):
        """Helper function to pad variable length RNN inputs with nans."""
        d = self.crop_or_pad_to
        difference = d - len(x)
        if difference < 0:
            return x[:d]
        pads = np.nan*np.ones((difference,) + x.shape[1:])
        return np.concatenate([x, pads])

class AugmentTimeVaryingSurvivalFeatures:
    def __init__(self,
                 only_censored_samples: bool = False,
                 probability: float = 0.5):
        self.rng = np.random.default_rng()
        self.only_censored_samples = only_censored_samples
        self.probability = probability

    def __call__(self, x: pd.DataFrame) -> pd.DataFrame:
        total_samples = len(x)
        if total_samples <= 1:
            return x
        if self.only_censored_samples and (1 in x['event_at_time'].unique()):
            return x
        # only sample with a certain probability
        if self.rng.random() > self.probability:
            return x
        new_length = self.rng.integers(1, total_samples)
        new_sample = x.iloc[:new_length].copy()
        return new_sample

In [4]:
test_data = FTPredDataset(ftpred_test)

In [ ]:
pmfs_preds = pd.read_csv('private_data/ft_db_cbcts_dummy_vit-t_ftdet_ddh_predictions.csv')
# convert arrays stored like strings back to numpy arrays
# example: '[6.6736928e-04 4.2621732e-01 4.1996208e-03 1.1884256e-02 1.3904880e-02\n 4.3673418e-03 9.8422036e-02 3.4654122e-02 2.5868330e-02 9.9970307e-03\n 3.1428116e-03 1.4601413e-02 2.3643047e-02 2.2490991e-03 9.6684275e-03\n 4.1013405e-02 6.3656536e-03 7.4682035e-03 2.3983167e-02 1.9348502e-03\n 9.6880198e-03 1.9428557e-02 2.5716587e-03 8.8854355e-04 2.8670928e-03\n 8.6026238e-03 1.3066637e-02 5.9948413e-04 1.8292485e-03 1.7134640e-02\n 6.9873338e-03 8.9911261e-04 6.3063647e-04 1.2396116e-02 1.3557329e-02\n 1.5661104e-03 1.2569805e-02 3.1658693e-04 6.8215027e-02 1.5204538e-02\n 1.5467617e-03 7.5364491e-04 8.6710006e-03 3.7588924e-04 1.2248857e-03\n 4.2274538e-03 9.9289389e-03]'
def convert_string_to_array(s):
    # since there are no commas and there are \n, we can split by whitespace and then convert to float
    return np.array([float(x) for x in s.replace('\n', ' ').replace('[', '').replace(']', '').split()])
pmfs_preds['pmf'] = pmfs_preds['pmf'].apply(convert_string_to_array)

0.00066736928